# RK-1 順位の確認 + EX-1 要素の探索（DP4・DP5・DP7・DP9 の記録、GPU）

> ## 🔑 先に許可しておくシークレット
> - **`GITHUB_TOKEN`（必須）**：リポジトリを決まったコミットで取ってきます。
> - **`HF_TOKEN`（必須）**：Gemma は利用許可の要るモデルです（DP5・DP7 と同じトークン）。
>
> 左の鍵のマークで「ノートブックからのアクセス」をオンにしてから「すべてのセルを実行」してね。

どちらも**事後の分析**です（計画は計算の前に凍結：`notes/rk1_tau_rank_plan.md` 2026-10-18 17:41 JST、`notes/ex1_feature_sweep_spec.md` 17:49 JST）。新しい続きは生成しません。**Claude の API は使いません**。

- Drive に残っている DP4・DP5・DP7・DP9 の記録を、それぞれ**回したときと同じコード**で読み直し、記録した結果と合ったときだけ使います。
- **RK-1**：DP7 と同じモデル・同じ計算で土台の道をなぞり、各位置で「確率 10⁻⁶ の語が何番目の候補だったか」を数えます。記録と合わなければ順位は使いません。
- **EX-1**：3 つのモデルで土台の道を 1 回ずつ通し、各位置の 20 の要素を計算して、分かれ道との関係を全部表にします。
- 結果は `DECISION_POINTS/rk1/RK1_RESULT.json` と `DECISION_POINTS/ex1/EX1_RESULT.json` に**数だけ**書きます。問題の中身・語の文字は表示しません（語の種類は分類だけ）。

**使い方**：**GPU（L4）**のランタイム（ハイメモリは不要）。「すべてのセルを実行」で 40〜80 分（モデル 3 つのダウンロードを含む）。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。
- 通知の頭には 🚀／⚠／🎉 が付きます。

In [ ]:
# Cell 1 — everything: clone, models, re-read records, RK-1 (GPU), EX-1 (GPU), analyses, results.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'RK-1 順位の確認 + EX-1 要素の探索（DP4・DP5・DP7・DP9、GPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'GodPork - Pork RK1 EX1', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'RK1_EX1'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('⚠ 止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイムにしてね（L4）"
GPU_NAME = gpu.stdout.strip().split(",")[0]
print("GPU:", GPU_NAME)

EXPECTED_HEAD = "a92192f5efe1fb11e7bf6008631e244c4545f3ab"
EXPS = json.loads(r"""{"dp4": ["dp4_weighting", "dp4_runner", ["A", "B"], "dp4/DP4_1B_{h}_20261008", "b64a0065590662c2270d110c80ff095f41f80f5e", "olmo", ["q_alt"]], "dp5": ["dp5_gemma", "dp5_runner", ["A", "B", "C"], "dp5/DP5_GEMMA_{h}_20261008", "fd5b17566df21a1c0522afa53b97f4af8b42d552", "gemma", ["q_alt"]], "dp7": ["dp7_matched", "dp7_runner", ["A", "B"], "dp7/DP7_GEMMA_{h}_20261010", "5592aa76bdd1d0576af563f241f2e31c96d33511", "gemma", ["q_swapt", "matched"]], "dp9": ["dp9_qwen", "dp9_runner", ["A", "B", "C", "D"], "dp9/DP9_QWEN_{h}_20261017", "43ef025fdedc964048a596536ce180139e4f549f", "qwen", ["q_alt", "q_swapt", "matched"]]}""")
COLS = json.loads(r"""["row", "template", "t", "entropy", "digit", "p1", "q_keep", "q_swap"]""")
MODELS = json.loads(r"""{"olmo": ["allenai/OLMo-2-0425-1B-Instruct", "48d788eca847d4d7548f375ad03d3c9312f6139e", "float16", null], "gemma": ["google/gemma-2-2b-it", "299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8", "bfloat16", "eager"], "qwen": ["Qwen/Qwen2.5-7B-Instruct", "a09a35458c702b33eeacc393d103063234e8bc28", "bfloat16", "sdpa"]}""")
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
WORK = Path("/content/rk1ex1_work")
MDIR = Path("/content/models")
for name, (_m, _r, parts, fmt, _c, _k, _x) in EXPS.items():
    for h in parts:
        assert (THEME / fmt.format(h=h) / "RUN_IDENTITY.json").exists(), f"{name} の実行フォルダ {h} が見つかりません"

def secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v else None
token = secret("GITHUB_TOKEN")
assert token, "Colab のシークレットに GITHUB_TOKEN を登録してね（ノートブックへのアクセスもオン）。"
hf = secret("HF_TOKEN") or os.environ.get("HF_TOKEN")
assert hf, "Colab のシークレットに HF_TOKEN を登録してね（Gemma は利用許可の要るモデル）。"
os.environ["HF_TOKEN"] = hf; hf = None
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
token = None
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
env.pop("GITHUB_TOKEN", None)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "huggingface_hub", "pyarrow", "scipy"],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
sys.path.insert(0, str(REPO))
from pork import rk1_rank as R, ex1_sweep as E, dp8x_explore as X
assert R.plan_doc_sha256(REPO) == R.PLAN_DOC_SHA256, "RK-1 の計画が凍結したものと違います"
assert E.spec_doc_sha256(REPO) == E.SPEC_DOC_SHA256, "EX-1 の計画が凍結したものと違います"
print("head", head[:12])

from huggingface_hub import snapshot_download
for key, (mid, rev, _dt, _at) in MODELS.items():
    snapshot_download(mid, revision=rev, local_dir=str(MDIR / key), token=os.environ["HF_TOKEN"],
                      allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model", "*.txt"])
    print("model ready:", key)

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
(WORK / "extract.py").write_text(r"""import importlib, json, sys, hashlib
from pathlib import Path
import numpy as np
spec = json.loads(sys.argv[1])
d = importlib.import_module("pork." + spec["module"]); rr = importlib.import_module("pork." + spec["runner"])
from pork import dp2_real_forks as d2
from pork import kv_cached_backend as kv
if spec["name"] == "dp9":
    rows_raw, _ = d2.load_rows_from_hub(); split = d.parts(); part_key = "part"
else:
    rows_raw, _ = d.load_rows_from_hub(); split = d.halves(); part_key = "half"
items = d.select_items(rows_raw); del rows_raw
items_sha = d.items_sha256(items)
cases, commits, idents = [], {}, {}
for h in spec["parts"]:
    b = Path(spec["base_fmt"].format(h=h))
    ident = json.loads((b / "RUN_IDENTITY.json").read_text())
    assert ident["items_sha256"] == items_sha and ident[part_key] == h and ident["gate_doc_sha256"] == d.GATE_DOC_SHA256, h
    assert not any(b.glob("RUN_FAILED*.txt")), h
    cases += rr.load_cases(split[h], units_dir=b / "units", code_commit=ident["code_commit"], items_hash=items_sha)
    commits[h] = ident["code_commit"]; idents[h] = ident
cases = sorted(cases, key=lambda c: c["row"])
assert [c["row"] for c in cases] == sorted(r for h in spec["parts"] for r in split[h])
analysis = json.loads(json.dumps(d.analyze(cases), sort_keys=True))
tab = d.position_table(cases)
out = Path(spec["out"]); out.mkdir(parents=True, exist_ok=True)
np.savez(out / "table.npz", **{k: np.asarray(tab[k]) for k in spec["cols"]})
(out / "analysis.json").write_text(json.dumps(analysis, sort_keys=True))
(out / "identity.json").write_text(json.dumps({"items_sha256": items_sha, "code_commits": commits,
                                               "gate_doc_sha256": d.GATE_DOC_SHA256, "schema": d.SCHEMA}, sort_keys=True))
import transformers
tok = transformers.AutoTokenizer.from_pretrained(spec["model_dir"])
ident0 = idents[spec["parts"][0]]
base = []
for c in cases:
    b = c["base"]
    if b["prediction"] is None or not b["answer_token"]:
        continue
    a, r = int(b["answer_token"]), int(c["row"])
    prompt = [int(x) for x in kv.chat_prompt_ids(tok, d2.user_message(items[r]["question"]))]
    if "prompt_token_count" in b:
        assert len(prompt) == int(b["prompt_token_count"]), ("prompt length differs", r)
    if "prompt_ids_sha256" in ident0:
        assert hashlib.sha256(json.dumps(prompt).encode()).hexdigest() == ident0["prompt_ids_sha256"][r], ("prompt hash", r)
    rec = {"row": r, "template": int(c["template"]), "a": a, "points": [int(x) for x in d.points(a)], "prompt_ids": prompt,
           "token_ids": [int(x) for x in b["token_ids"][:a + 1]], "top2": [int(x) for x in b["top2"][:a]],
           "entropy": [float(x) for x in b["entropy"][:a]]}
    if "tau_token" in b:
        rec["tau_token"] = [int(x) for x in b["tau_token"][:a]]; rec["tau_p"] = [float(x) for x in b["tau_p"][:a]]
    base.append(rec)
(out / "base.json").write_text(json.dumps(base))
print("extracted", spec["name"], "positions", len(tab["row"]), "items with a base path", len(base))
""")
(WORK / "rk1_gpu.py").write_text(r"""import importlib.util, json, sys
from pathlib import Path
import numpy as np
spec = json.loads(sys.argv[1])
from pork import dp7_matched as d7, dp7_runner as rr, kv_cached_backend as kv, pork_t_v1 as pork_t
from pork.ct1_runner import load_gemma, torch_fns
u = importlib.util.spec_from_file_location("rk1_core", spec["core"]); C = importlib.util.module_from_spec(u); u.loader.exec_module(C)
torch, tok, model = load_gemma(spec["model_dir"], None, d7.DEVICE)
assert str(model.dtype) == "torch.bfloat16" and model.config._attn_implementation == "eager"
eos = kv.eos_ids(tok, model); excluded = rr.excluded_ids(tok, eos)
ident = json.loads(Path(spec["ident"]).read_text())
assert ident["tau_excluded_ids"] == excluded and ident["eos_ids"] == list(eos), "DP7 identity differs (EOS / excluded ids)"
Bn = pork_t.backend("numpy", "cpu")
single, _ = torch_fns(model, torch, d7.DEVICE)
base = json.loads(Path(spec["base_json"]).read_text())
pos, top1_same = [], []
for rec in base:
    step = single(); past = None; new = tuple(rec["prompt_ids"]); toks = rec["token_ids"]; pts = set(rec["points"])
    for k in range(rec["a"]):
        logits, past = step(new, past)
        lg = logits.detach().cpu().numpy()
        top1_same.append(int(np.argmax(lg)) == int(toks[k]))
        if k in pts:
            r = C.rank_record(lg, toks[k], rec["tau_token"][k], rec["tau_p"][k], excluded, d7.tau_choice, Bn)
            pos.append({"row": rec["row"], "t": k, **r})
        new = (int(toks[k]),)
Path(spec["out"]).write_text(json.dumps({"positions": pos, "top1_same": top1_same}))
print("RK-1 ranks", len(pos), "positions,", len(top1_same), "steps")
""")
(WORK / "ex1_gpu.py").write_text(r"""import json, sys
from pathlib import Path
import numpy as np
spec = json.loads(sys.argv[1])
sys.path.insert(0, spec["repo"])
from pork import ex1_sweep as E, kv_cached_backend as kv
import torch, transformers
tok = transformers.AutoTokenizer.from_pretrained(spec["model_dir"])
kw = {"torch_dtype": getattr(torch, spec["dtype"])}
if spec["attn"]:
    kw["attn_implementation"] = spec["attn"]
model = transformers.AutoModelForCausalLM.from_pretrained(spec["model_dir"], **kw).to("cuda")
model.eval()
eos = kv.eos_ids(tok, model); excluded = sorted(set(int(x) for x in eos) | set(int(x) for x in tok.all_special_ids))
cache = {}
def kind(i):
    if i not in cache:
        cache[i] = E.classify(tok.decode([int(i)]))
    return cache[i]
for exp in spec["exps"]:
    base = json.loads(Path(exp["base_json"]).read_text())
    rows = []
    for rec in base:
        a = rec["a"]; toks = rec["token_ids"]
        ids = rec["prompt_ids"] + toks[:a]
        with torch.no_grad():
            out = model(input_ids=torch.tensor([ids], device="cuda"), logits_to_keep=a + 1, use_cache=False)
        L = out.logits[0, :-1].float()
        assert L.shape[0] == a, (L.shape, a)
        kinds = [kind(x) for x in toks[:a]]
        bk, bn = [k for k, _ in kinds], [n for _, n in kinds]
        for t in rec["points"]:
            f = E.dist_features(L[t].cpu().numpy(), toks[t], excluded)
            kf = E.kind_features(t, a, bk, bn, kind(rec["top2"][t])[0])
            rows.append({"row": rec["row"], "t": t, **f, **kf, "top1_same": int(f["top1"]) == int(toks[t]),
                         "entropy_rec": rec["entropy"][t]})
        del out, L
    Path(exp["out"]).write_text(json.dumps(rows))
    print("EX-1 features", exp["name"], len(rows), "positions")
""")
shutil.copy(REPO / "pork" / "rk1_core.py", WORK / "rk1_core.py")
def run(script, spec, cwd, pythonpath):
    p = subprocess.run([sys.executable, str(WORK / script), json.dumps(spec)], cwd=str(cwd),
                       env=dict(os.environ, PYTHONPATH=str(pythonpath)), capture_output=True, text=True)
    assert p.returncode == 0, script + " に失敗しました:\n" + p.stderr[-3000:]
    print(p.stdout.strip().splitlines()[-1])

# --- step 2: re-read each experiment at its run commit (checked against the recorded result)
TABLES, CHECKS, STORED, WT = {}, {}, {}, {}
for name, (module, runner, parts, fmt, commit, mkey, extra) in EXPS.items():
    wt = WORK / ("wt_" + name); WT[name] = wt
    subprocess.run(["git", "-C", str(REPO), "worktree", "add", "-q", "--detach", str(wt), commit], check=True)
    o = WORK / ("out_" + name)
    run("extract.py", {"name": name, "module": module, "runner": runner, "parts": parts, "base_fmt": str(THEME / fmt),
                       "out": str(o), "cols": COLS + extra, "model_dir": str(MDIR / mkey)}, wt, wt)
    analysis = json.loads((o / "analysis.json").read_text()); ident = json.loads((o / "identity.json").read_text())
    stored = json.loads((REPO / "reports" / (name + "_RESULT.json")).read_text())
    st = {k: v for k, v in stored.items() if k != "result_sha256"}
    assert hashlib.sha256(json.dumps(st, sort_keys=True).encode()).hexdigest() == stored["result_sha256"], name + " stored"
    assert set(ident["code_commits"].values()) == {commit}, name
    assert ident["items_sha256"] == stored["items_sha256"] and ident["gate_doc_sha256"] == stored["gate_doc_sha256"], name
    rebuilt = {"schema": ident["schema"], "analysis": analysis, "items_sha256": ident["items_sha256"],
               "gate_doc_sha256": ident["gate_doc_sha256"], "versions": stored["versions"]}
    if "code_commit" in stored:
        assert stored["code_commit"] == commit, name
        rebuilt["code_commit"] = commit
    else:
        assert ident["code_commits"] == stored["code_commits"], name
        rebuilt["code_commits"] = ident["code_commits"]
    exact = hashlib.sha256(json.dumps(rebuilt, sort_keys=True).encode()).hexdigest() == stored["result_sha256"]
    diffs = [] if exact else X.same_numbers(analysis, stored["analysis"], tol=1e-9)
    assert exact or not diffs, f"{name}: 記録した結果と合いません（{len(diffs)} か所）— Claude に見せてね"
    CHECKS[name] = {"result_sha256_exact": exact, "within_1e-9": True, "stored_result_sha256": stored["result_sha256"]}
    print(name, "記録と照合:", "完全に一致" if exact else "丸めの違いだけ（1e-9 以内）")
    z = __import__("numpy").load(o / "table.npz")
    TABLES[name] = {k: z[k] for k in z.files}
    STORED[name] = stored["analysis"]

# --- step 3: RK-1 on the GPU, in DP7's worktree
dp7_ident = THEME / EXPS["dp7"][3].format(h="A") / "RUN_IDENTITY.json"
run("rk1_gpu.py", {"model_dir": str(MDIR / "gemma"), "base_json": str(WORK / "out_dp7" / "base.json"),
                   "core": str(WORK / "rk1_core.py"), "ident": str(dp7_ident), "out": str(WORK / "rk1_raw.json")},
    WT["dp7"], WT["dp7"])

# --- step 4: EX-1 on the GPU, one model per process
for key, (mid, rev, dtype, attn) in MODELS.items():
    exps = [{"name": n, "base_json": str(WORK / ("out_" + n) / "base.json"), "out": str(WORK / ("ex1_" + n + ".json"))}
            for n, v in EXPS.items() if v[5] == key]
    run("ex1_gpu.py", {"repo": str(REPO), "model_dir": str(MDIR / key), "dtype": dtype, "attn": attn, "exps": exps},
        REPO, REPO)

# --- step 5: analyses
import numpy as np, scipy
raw = json.loads((WORK / "rk1_raw.json").read_text())
RK_CHECK = R.checks(raw["top1_same"], [p["tau_same"] for p in raw["positions"]], [p["p_rel_diff"] for p in raw["positions"]])
rank_at = {(p["row"], p["t"]): p["rank"] for p in raw["positions"]}
t7 = TABLES["dp7"]
if RK_CHECK["pass"]:
    m = t7["matched"] == 1
    tab7 = {k: t7[k][m] for k in ("row", "template", "t", "entropy", "digit")}
    tab7["e_tau"] = (t7["q_swapt"] - t7["q_keep"])[m]
    ranks = np.asarray([rank_at[(int(r), int(t))] for r, t in zip(tab7["row"], tab7["t"])])
    rk_a = R.analyze(tab7, ranks); rk_b = R.analyze(tab7, ranks)
    assert R.result_sha(rk_a) == R.result_sha(rk_b), "RK-1 の集計が再現しません"
else:
    rk_a = None
rk_res = {"schema": R.SCHEMA, "checks": RK_CHECK, "analysis": rk_a, "reread_dp7": CHECKS["dp7"], "code_commit": EXPECTED_HEAD,
          "plan_doc_sha256": R.PLAN_DOC_SHA256, "gpu_name": GPU_NAME,
          "versions": {"numpy": np.__version__, "scipy": scipy.__version__}}
rk_res["result_sha256"] = R.result_sha(rk_res)

FEATS, DIST_OK, EX_CHECKS = {}, {}, {}
for name in EXPS:
    rows = json.loads((WORK / ("ex1_" + name + ".json")).read_text())
    at = {(r["row"], r["t"]): r for r in rows}
    tab = TABLES[name]
    keys = [(int(r), int(t)) for r, t in zip(tab["row"], tab["t"])]
    assert all(k in at for k in keys), name + ": 要素のない位置があります"
    ck = E.recompute_check([at[k]["top1_same"] for k in keys], [at[k]["entropy_re"] for k in keys], tab["entropy"])
    EX_CHECKS[name] = ck; DIST_OK[name] = ck["pass"]
    f = {g: np.asarray([float(at[k][g]) for k in keys]) for g in E.DIST_FEATURES + E.KIND_FEATURES + E.POS_FEATURES + E.REFERENCE}
    f["entropy"] = np.asarray(tab["entropy"], float)
    if name == "dp7":
        f["tau_rank"] = np.asarray([np.log10(rank_at[k]) if (RK_CHECK["pass"] and k in rank_at) else np.nan for k in keys])
    FEATS[name] = f
ex_a = E.analyze_all(TABLES, FEATS, DIST_OK); ex_b = E.analyze_all(TABLES, FEATS, DIST_OK)
assert E.result_sha(ex_a) == E.result_sha(ex_b), "EX-1 の集計が再現しません"
ex_res = {"ex1": ex_a, "recompute_checks": EX_CHECKS, "reread_checks": CHECKS, "rk1_checks_for_dp7_tau_rank": RK_CHECK,
          "code_commit": EXPECTED_HEAD, "spec_doc_sha256": E.SPEC_DOC_SHA256, "gpu_name": GPU_NAME,
          "versions": {"numpy": np.__version__, "scipy": scipy.__version__}}
ex_res["result_sha256"] = E.result_sha(ex_res)
for sub, fname, obj in (("rk1", "RK1_RESULT.json", rk_res), ("ex1", "EX1_RESULT.json", ex_res)):
    (THEME / sub).mkdir(parents=True, exist_ok=True)
    tmp = THEME / sub / (fname + ".tmp")
    tmp.write_text(json.dumps(obj, sort_keys=True, indent=1))
    os.replace(tmp, THEME / sub / fname)

f3 = lambda v: "  —  " if v is None else f"{v:.3f}"
print("\n[RK-1] 照合:", "合格" if RK_CHECK["pass"] else "不合格（順位は使いません）",
      f"（1 位 {RK_CHECK['top1_share']:.4f}・w_tau {RK_CHECK['tau_share']:.4f}・確率 {RK_CHECK['p_within_1pct_share']:.4f}）")
if rk_a:
    for k in rk_a["K1_rank_by_entropy_tertile"]:
        print(f"  エントロピーの三分位 {k['tertile']}: 順位の中央値 {k['rank_median']:.0f}（10 位以内 {k['share_rank_le_10']:.2f}）")
    print("  K4 順位が低い組:", f3(rk_a["K4_low_rank"]["within_item_auroc"]), rk_a["K4_low_rank"]["ci"])
    print("  K4 順位が高い組:", f3(rk_a["K4_high_rank"]["within_item_auroc"]), rk_a["K4_high_rank"]["ci"])
    print("  読み方:", rk_a["reading"])
print("\n[EX-1] 確率の照合:", {n: c["pass"] for n, c in EX_CHECKS.items()})
print("  候補:", len(ex_a["rules"]["candidates"]), "| 探索の中で再現:", len(ex_a["rules"]["replicated_in_exploration"]))
for c in ex_a["rules"]["replicated_in_exploration"]:
    print("   -", c["feature"], c["view"], c["direction"])
print("\n保存: DECISION_POINTS/rk1/RK1_RESULT.json と DECISION_POINTS/ex1/EX1_RESULT.json")
print("✅ 終わり。Claude に知らせてね")
notify('🎉 終わりました：RK-1（' + ('照合 合格・' + rk_a['reading']['K4'] if rk_a else '照合 不合格') + '）・EX-1（候補 ' + str(len(ex_a['rules']['candidates'])) + '・再現 ' + str(len(ex_a['rules']['replicated_in_exploration'])) + '）', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()